> **TrustBreast — Notebook 6 (portability check).** Section 4.10: the same leakage-free pipeline on the Breast Cancer Coimbra cohort. Needs no model files.

# 06 — Pipeline portability: Breast Cancer Coimbra (Section 4.10)

## How to run
1. Colab → **Runtime → Change runtime type → CPU** (default; not GPU)
2. **Runtime → Run all** (Drive is not needed — the data are downloaded automatically from UCI)
3. Time: about **5–10 minutes**
4. Check the output of **STEP 5 — RESULTS SUMMARY**.

## What it does
- 116 patients (64 cancer, 52 controls), 9 anthropometric/blood features.
- Exactly the same pipeline as for WBCD: scaler + SMOTE inside each fold, RF + XGBoost + DNN, soft voting (0.5), per-fold seeds.
- 5-fold stratified CV (the cohort is small); it runs **twice** — both runs should give identical results.
- Cross-conformal (α = 0.05) on the out-of-fold probabilities — marginal and class-conditional coverage.

In [ ]:
# Colab: clone the repository (it contains the models/ folder). On a local Jupyter install this cell does nothing.
import os
if os.path.exists('/content') and not os.path.isdir('models') and not os.path.isdir('../models'):
    !git clone -q https://github.com/Iqra672-ai/TrustBreast.git /content/TrustBreast
    %cd /content/TrustBreast
    !pip -q install -r requirements.txt


In [ ]:
# ---------- STEP 0: determinism ----------
import os
os.environ['PYTHONHASHSEED'] = '42'; os.environ['TF_DETERMINISTIC_OPS'] = '1'; os.environ['TF_CUDNN_DETERMINISTIC'] = '1'
import random, numpy as np, tensorflow as tf
SEED = 42
tf.keras.utils.set_random_seed(SEED)
try: tf.config.experimental.enable_op_determinism()
except Exception: pass
print("TF", tf.__version__, "| determinism on")


## STEP 1 — Data (UCI Breast Cancer Coimbra)

In [ ]:
import pandas as pd, numpy as np
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/00451/dataR2.csv"
import os
local = next((f for f in ['data/coimbra_dataR2.csv', '../data/coimbra_dataR2.csv', 'dataR2.csv'] if os.path.exists(f)), None)
try:
    df = pd.read_csv(local if local else URL)
    print("Data source:", local if local else URL)
except Exception as e:
    print("Direct download failed, trying ucimlrepo:", str(e)[:60])
    import subprocess; subprocess.run(['pip', '-q', 'install', 'ucimlrepo'])
    from ucimlrepo import fetch_ucirepo
    r = fetch_ucirepo(id=451); df = pd.concat([r.data.features, r.data.targets], axis=1)
target = 'Classification'
X_c = df.drop(columns=[target]).astype(float).values
y_c = (df[target].values == 2).astype(int)          # 1 = breast cancer, 0 = healthy control
FEATS_C = list(df.drop(columns=[target]).columns)
print(f"Coimbra: {len(y_c)} patients | cancer {y_c.sum()} | controls {(y_c == 0).sum()} | features {len(FEATS_C)}: {FEATS_C}")
assert len(y_c) == 116 and y_c.sum() == 64


## STEP 2 — Leakage-free 5-fold CV of the full ensemble (same pipeline as WBCD)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, recall_score, confusion_matrix
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

N_FOLDS = 5
def build_dnn(dim, seed):
    tf.keras.utils.set_random_seed(seed)
    m = tf.keras.Sequential([tf.keras.layers.Input(shape=(dim,))])
    for u, dr, reg in [(256, .3, 5e-4), (128, .3, 5e-4), (64, .2, None)]:
        m.add(tf.keras.layers.Dense(u, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(reg) if reg else None))
        m.add(tf.keras.layers.BatchNormalization()); m.add(tf.keras.layers.Dropout(dr))
    m.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    m.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy'); return m

def run_cv(X, y, seed=SEED):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    fold_acc, oof = [], np.zeros(len(y))
    for fold, (tr, va) in enumerate(skf.split(X, y), 1):
        sc = MinMaxScaler().fit(X[tr]); Xtr, Xva = sc.transform(X[tr]), sc.transform(X[va])
        Xtr, ytr = SMOTE(random_state=seed).fit_resample(Xtr, y[tr])          # inside the fold only
        rf  = RandomForestClassifier(n_estimators=500, random_state=seed, n_jobs=1).fit(Xtr, ytr)
        xgb = XGBClassifier(n_estimators=500, max_depth=4, learning_rate=0.01, subsample=0.9,
                            colsample_bytree=0.9, random_state=seed, n_jobs=1,
                            eval_metric='logloss', verbosity=0).fit(Xtr, ytr)
        dnn = build_dnn(Xtr.shape[1], seed + fold)
        dnn.fit(Xtr, ytr, epochs=80, batch_size=16, verbose=0, validation_split=0.15,
                callbacks=[tf.keras.callbacks.EarlyStopping(patience=12, restore_best_weights=True)])
        p = (rf.predict_proba(Xva)[:, 1] + xgb.predict_proba(Xva)[:, 1] + dnn.predict(Xva, verbose=0).ravel()) / 3
        oof[va] = p; fold_acc.append(accuracy_score(y[va], (p >= .5).astype(int)))
        print(f"  fold {fold}: accuracy {fold_acc[-1]*100:.2f}%")
    return np.array(fold_acc), oof

print("Run 1:"); acc1, oof1 = run_cv(X_c, y_c)
print("Run 2 (reproducibility check):"); acc2, oof2 = run_cv(X_c, y_c)
REPRO = np.array_equal(acc1, acc2) and float(np.max(np.abs(oof1 - oof2))) < 1e-6
print("REPRODUCIBLE ✅" if REPRO else f"NOT reproducible ❌ (max prob diff {np.max(np.abs(oof1-oof2)):.2e})")


## STEP 3 — Metrics (out-of-fold, threshold 0.5)

In [ ]:
pred = (oof1 >= .5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_c, pred).ravel()
rs = np.random.RandomState(42); boot = []
for _ in range(2000):
    b = rs.randint(0, len(y_c), len(y_c)); boot.append((pred[b] == y_c[b]).mean())
M = dict(acc_mean=acc1.mean()*100, acc_sd=acc1.std(ddof=1)*100, pooled=(pred == y_c).mean()*100,
         ci=np.percentile(boot, [2.5, 97.5])*100, auc=roc_auc_score(y_c, oof1), f1=f1_score(y_c, pred),
         sens=tp/(tp+fn), spec=tn/(tn+fp), cm=(tn, fp, fn, tp))
print(f"Accuracy {M['acc_mean']:.2f} ± {M['acc_sd']:.2f} | pooled {M['pooled']:.2f} | CI {M['ci'][0]:.2f}–{M['ci'][1]:.2f}")
print(f"AUC {M['auc']:.4f} | F1 {M['f1']:.4f} | sensitivity {M['sens']:.4f} | specificity {M['spec']:.4f} | TN FP FN TP = {M['cm']}")


## STEP 4 — Cross-conformal coverage (α = 0.05, out-of-fold ensemble probabilities)

In [ ]:
import math
scores = np.where(y_c == 1, 1 - oof1, oof1)
def tau_of(sc, a):
    sc = np.sort(sc); n = len(sc); k = math.ceil((n + 1) * (1 - a)); return sc[min(k, n) - 1], k, n
CC = {}
for a in (0.05, 0.10):
    t, k, n = tau_of(scores, a)
    tb, kb, nb = tau_of(scores[y_c == 0], a); tm, km, nm = tau_of(scores[y_c == 1], a)
    inc0, inc1 = oof1 <= t, (1 - oof1) <= t
    cov = np.mean(np.where(y_c == 1, inc1, inc0)); size = inc0.astype(int) + inc1.astype(int)
    cov_c = np.mean((1 - oof1[y_c == 1]) <= t); cov_h = np.mean(oof1[y_c == 0] <= t)
    mcov_c = np.mean((1 - oof1[y_c == 1]) <= tm); mcov_h = np.mean(oof1[y_c == 0] <= tb)
    CC[a] = dict(t=t, cov=cov, cov_c=cov_c, cov_h=cov_h, tb=tb, tm=tm, mcov_c=mcov_c, mcov_h=mcov_h,
                 sets=(int(np.sum(size == 1)), int(np.sum(size == 2)), int(np.sum(size == 0))),
                 sat=(kb >= nb, km >= nm))
    print(f"alpha={a:.2f}: tau {t:.4f} | marginal {cov*100:.2f}% (cancer {cov_c*100:.2f}%, controls {cov_h*100:.2f}%) "
          f"| sets {CC[a]['sets']} | Mondrian cancer {mcov_c*100:.2f}% controls {mcov_h*100:.2f}% | saturated {CC[a]['sat']}")


# ---------- out-of-fold (fold-held-out) conformal evaluation ----------
import math, numpy as np
def kth(sc, alpha):
    sc = np.sort(sc); n = len(sc); k = math.ceil((n + 1) * (1 - alpha)); return sc[min(k, n) - 1], k, n

def conformal_report(y, p, fold, alpha, label):
    """y: labels (1 = positive class), p: out-of-fold P(positive), fold: fold id of each patient."""
    y = np.asarray(y).astype(int); p = np.asarray(p, float); fold = np.asarray(fold)
    s = np.where(y == 1, 1 - p, p)                       # nonconformity score, Eq. (4)
    # (a) in-sample: threshold and coverage from the same 569 scores (what the paper reported)
    t, k, n = kth(s, alpha); ins = np.mean(s <= t)
    # (b) out-of-fold: for each fold, threshold from the OTHER folds only, applied to this fold
    cov = np.zeros(len(y), bool); size = np.zeros(len(y), int)
    covM = np.zeros(len(y), bool)
    for f in np.unique(fold):
        te, ca = fold == f, fold != f
        tj, _, _ = kth(s[ca], alpha)                      # marginal threshold
        tb, _, _ = kth(s[ca & (y == 0)], alpha)           # Mondrian thresholds
        tm, _, _ = kth(s[ca & (y == 1)], alpha)
        inc0, inc1 = p[te] <= tj, (1 - p[te]) <= tj
        cov[te] = np.where(y[te] == 1, inc1, inc0); size[te] = inc0.astype(int) + inc1.astype(int)
        covM[te] = np.where(y[te] == 1, (1 - p[te]) <= tm, p[te] <= tb)
    print(f"{label}  alpha={alpha:.2f}")
    print(f"   in-sample (paper so far): tau {t:.4f}  index {k}/{n}  coverage {ins*100:.2f}%  (= index/n by construction)")
    print(f"   OUT-OF-FOLD marginal coverage {cov.mean()*100:.2f}%  | positive {cov[y==1].mean()*100:.2f}%  negative {cov[y==0].mean()*100:.2f}%")
    print(f"   OUT-OF-FOLD sets single/ambiguous/empty = {(size==1).sum()}/{(size==2).sum()}/{(size==0).sum()}")
    print(f"   OUT-OF-FOLD Mondrian coverage  positive {covM[y==1].mean()*100:.2f}%  negative {covM[y==0].mean()*100:.2f}%")
    return dict(ins=ins, cov=cov.mean(), cpos=cov[y==1].mean(), cneg=cov[y==0].mean(),
                sets=((size==1).sum(), (size==2).sum(), (size==0).sum()), mpos=covM[y==1].mean(), mneg=covM[y==0].mean())

fold_c = np.zeros(len(y_c), int)
for f, (_, va) in enumerate(StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED).split(X_c, y_c), 1): fold_c[va] = f
OOF = {a: conformal_report(y_c, oof1, fold_c, a, 'Coimbra') for a in (0.05, 0.10)}


## STEP 5 — ★ RESULTS SUMMARY (only this cell's output is needed)

In [ ]:
c = CC[0.05]
print("="*74 + "\nCOIMBRA PORTABILITY CHECK — Section 4.10\n" + "="*74)
print(f"Reproducible across two runs : {'✅' if REPRO else '❌'}")
print(f"Accuracy (5-fold)            : {M['acc_mean']:.2f} ± {M['acc_sd']:.2f}%   [paper: 70.62 ± 14.35]")
print(f"Pooled out-of-fold accuracy  : {M['pooled']:.2f}%   CI {M['ci'][0]:.2f}–{M['ci'][1]:.2f}   [paper: 70.69, 62.07–79.31]")
print(f"AUC / F1                     : {M['auc']:.2f} / {M['f1']:.2f}   [paper: 0.80 / 0.73]")
print(f"Sensitivity / specificity    : {M['sens']:.2f} / {M['spec']:.2f}   [paper: 0.73 / 0.67]")
o = OOF[0.05]
print(f"Cross-conformal α=0.05 (held-out folds): marginal {o['cov']*100:.1f}%, cancer {o['cpos']*100:.1f}%, controls {o['cneg']*100:.1f}%   [paper: 95.7 / 98.4 / 92.3]")
print(f"  Mondrian (held-out folds)  : cancer {o['mpos']*100:.1f}%, controls {o['mneg']*100:.1f}%  | sets {tuple(int(v) for v in o['sets'])}   [paper: 96.9 / 92.3 | (38, 78, 0)]")
print("\nNumbers in [paper: ...] are from a Colab CPU-runtime run (Section 4.10). A GPU runtime can differ in the last decimal.")
